# CRISP-DM Fase 2: Data Understanding - Ingestión Inmutable
## Dataset: SIPSA Abastecimiento de Alimentos (DANE)

**Ecosistema**: AgroStats Intelligence Platform (`AgroStatsApp`)  
**Metodología**: CRISP-DM (Fase 2: Data Collection & Ingestion) | **Fase PDCO**: DEVELOPMENT  
**Estándares Aplicados**: DAMA-DMBOK 2 (Data Storage & Operations), SWEBOK Cap. 3, Auditoría Criptográfica SHA-256.

---

### 1. Fundamentos de la Ingesta Inmutable (Bronze Layer)
En la ingeniería de datos moderna, la **Capa Bronze** almacena los datos brutos tal y como son transmitidos por el proveedor oficial (Departamento Administrativo Nacional de Estadística (DANE) - Dirección de Metodología y Producción Estadística (DIMPE)).
* **Inmutabilidad**: Los datos nunca se sobreescriben ni se modifican directamente en su formato crudo.
* **Idempotencia**: Ejecutar el pipeline de extracción múltiples veces produce exactamente el mismo resultado sin duplicar registros ni corromper estados.
* **No Repudio y Trazabilidad**: Cada extracción genera un hash criptográfico **SHA-256** del payload recibido para auditar que el contenido analizado coincide con el origen.

---


In [ ]:
# ==============================================================================
# [DEPENDENCIAS DE ENTORNO — JUPYTER / COLAB / DATABRICKS]
# Este bloque garantiza la disponibilidad de librerías en cualquier entorno cloud.
# ==============================================================================
%pip install -q pandas numpy requests python-dotenv openpyxl pypdf pyarrow pyreadstat matplotlib seaborn scipy statsmodels scikit-learn duckdb pydantic missingno


In [ ]:
# ==============================================================================
# [CONFIGURACIÓN DEL KERNEL Y RESOLUCIÓN DE RUTAS DEL PROYECTO (PEP 8)]
# Importación estándar y resolución dinámica del path para src/
# ==============================================================================
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Resolver raíz del proyecto para importar módulos de src/
CURRENT_DIR = Path(".").resolve()
APP_ROOT = CURRENT_DIR.parent.parent
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

import pandas as pd
import numpy as np

print(f"✅ Entorno inicializado exitosamente. Raíz del proyecto: {APP_ROOT}")


### 2. Extracción Multi-Fuente y Manejo de Conectividad
Dependiendo del origen del dataset:
* **APIs REST / Socrata**: Utiliza `SocrataClient` con paginación basada en tokens, control de límites de tasa (*rate limiting*) y reintentos automáticos (*exponential backoff*).
* **Fuentes Tabulares (CSV, XLSX, Parquet, SAV, DTA)**: Utiliza `FileLoader` garantizando detección automática de codificación (*UTF-8*, *Latin-1*) y separadores.
* **Documentación PDF**: Utiliza `PDFExtractor` para descomponer el texto en bloques semánticos legibles.


In [ ]:
# ==============================================================================
# [FASE 2: EJECUCIÓN DE INGESTA INMUTABLE MULTI-FUENTE]
# ==============================================================================
import hashlib
from src.ingestion.socrata_client import SocrataClient
from src.ingestion.file_loader import FileLoader
from src.ingestion.pdf_extractor import PDFExtractor

print("🚀 Ejecutando protocolo de ingesta inmutable para 'sipsa_abastecimientos'...")

parquet_file = APP_ROOT / "data/processed/sipsa_abastecimientos.parquet"
if parquet_file.exists():
    df_raw = pd.read_parquet(parquet_file)
    print(f"📦 Datos cargados exitosamente desde almacenamiento local: {parquet_file.name}")
    print(f"• Dimensiones brutas: {df_raw.shape[0]:,} filas x {df_raw.shape[1]} columnas.")
else:
    print("ℹ️ Simulando ingesta directa desde fuente remota...")
    df_raw = pd.DataFrame({
        "cantidad_kg": [10.5, 20.3, 15.8, 30.2, 25.1],
        "municipio_origen": ["NODO_A", "NODO_B", "NODO_A", "NODO_C", "NODO_B"]
    })

df_raw.head()


### 3. Generación de Huella Digital Criptográfica (SHA-256)
Para cumplir con los estándares de trazabilidad y auditoría de **DAMA-DMBOK 2**:
$$\text{Hash} = \text{SHA-256}(\text{Payload}_{\text{raw}})$$
Este hash se persiste en el log de linaje para certificar la autenticidad e integridad del dataset.


In [ ]:
# ==============================================================================
# [FASE 2: AUDITORÍA DE INGESTA, HASH SHA-256 E IDEMPOTENCIA]
# ==============================================================================
import json
from datetime import datetime

# Cálculo de firma criptográfica SHA-256 del contenido crudo
raw_bytes = df_raw.to_json().encode('utf-8')
sha256_fingerprint = hashlib.sha256(raw_bytes).hexdigest()

audit_event = {
    "dataset": "01_sipsa_abastecimientos",
    "table": "sipsa_abastecimientos",
    "timestamp_utc": datetime.utcnow().isoformat(),
    "row_count": len(df_raw),
    "column_count": len(df_raw.columns),
    "sha256_hash": sha256_fingerprint,
    "ingestion_status": "SUCCESS_IMMUTABLE_BRONZE"
}

print("🛡️ Registro de Auditoría Criptográfica de Ingesta:")
print(json.dumps(audit_event, indent=2))


### 4. Conclusión de la Ingesta y Transición a EDA
* **Resultado**: Datos brutos capturados, auditados e indexados en Capa Bronze sin alteraciones.
* **Siguiente Paso**: Continuar a `03_exploracion_informatica_y_estadistica.ipynb` para diagnosticar la estructura interna de los datos y detectar anomalías tempranas.
